In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    get_mem_and_time_profiling,
)
from image_analysis_3D.featurization_utils.texture_utils import measure_3D_texture

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/mahotas/morph.py:315: SyntaxWarning: invalid escape sequence '\s'
  '''
/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/mahotas/features/texture.py:33: SyntaxWarning: invalid escape sequence '\|'
  '''
/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/mahotas/features/texture.py:158: SyntaxWarning: invalid escape sequence '\|'
  '''


In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "D11-2"
    patient = "NF0016_T1"
    channel = "DNA"
    compartment = "Cell"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)

In [3]:
channel_mapping = {
    "DNA": "405",
    "AGP": "488",
    "ER": "555",
    "Mito": "640",
    "BF": "TRANS",
    "Nuclei": "nuclei_",
    "Cell": "cell_",
    "Cytoplasm": "cytoplasm_",
    "Organoid": "organoid_",
}

In [4]:
start_time = time.time()
# get starting memory (cpu)
start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_mapping,
    image_set_name=well_fov,
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
output_texture_dict = measure_3D_texture(
    object_loader=object_loader,
    distance=3,  # distance in pixels 3 is what CP uses
)
final_df = pd.DataFrame(output_texture_dict)

final_df = final_df.pivot(
    index="object_id",
    columns="texture_name",
    values="texture_value",
)
final_df.reset_index(inplace=True)
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Granularity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
final_df.columns.name = None

output_file = pathlib.Path(
    output_parent_path
    / f"Texture_{compartment}_{channel}_{processor_type}_features.parquet"
)
output_file.parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(output_file)
final_df.head()

5it [00:28,  5.65s/it]


,image_set,object_id,Cell_DNA_Granularity_AngularSecondMoment-256-3,Cell_DNA_Granularity_Contrast-256-3,Cell_DNA_Granularity_Correlation-256-3,Cell_DNA_Granularity_DifferenceEntropy-256-3,Cell_DNA_Granularity_DifferenceVariance-256-3,Cell_DNA_Granularity_Entropy-256-3,Cell_DNA_Granularity_InformationMeasureOfCorrelation1-256-3,Cell_DNA_Granularity_InformationMeasureOfCorrelation2-256-3,Cell_DNA_Granularity_InverseDifferenceMoment-256-3,Cell_DNA_Granularity_SumAverage-256-3,Cell_DNA_Granularity_SumEntropy-256-3,Cell_DNA_Granularity_SumVariance-256-3,Cell_DNA_Granularity_Variance-256-3
0,D11-2,257,0.963582,4.957242,0.920379,0.180939,0.003769,0.280249,-0.634284,0.478627,0.985277,1.214971,0.243572,119.031410,30.997163
1,D11-2,514,0.997492,3.844221,0.736246,0.021109,0.003881,0.026621,-0.447783,0.121685,0.998806,0.140820,0.023065,24.653799,7.124505
2,D11-2,771,0.997416,2.205036,0.769142,0.021433,0.003881,0.027634,-0.456638,0.125480,0.998794,0.108295,0.023702,16.427546,4.658146
3,D11-2,1285,0.988736,2.904701,0.871442,0.067686,0.003851,0.099401,-0.612800,0.289772,0.995192,0.413271,0.083380,41.698333,11.150759
4,D11-2,1542,0.993221,3.669596,0.828219,0.040863,0.003868,0.057677,-0.611708,0.222422,0.997112,0.341080,0.051252,38.824740,10.623584


In [7]:
end_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2
end_time = time.time()
get_mem_and_time_profiling(
    start_mem=start_mem,
    end_mem=end_mem,
    start_time=start_time,
    end_time=end_time,
    feature_type="Texture",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU="CPU",
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Texture_CPU.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0016_T1
        Well and FOV: D11-2
        Feature type: Texture
        CPU/GPU: CPU
        Memory usage: 1802.07 MB
        Time elapsed:
        --- 64.49 seconds ---
        --- 1.07 minutes ---
        --- 0.02 hours ---
    


True